# DreamZero Video / Action timestep：解耦、当前 bug 与正确 couple

这个最小示例复现 `DreamZeroHead.forward()` 中 Action timestep 的构造逻辑，回答三个问题：

1. `decouple_video_action_noise=True` 时为什么 Action 是独立 `randint`；
2. `False` 时当前 `.repeat(1, 1, k)` 如何破坏 batch 轴并触发随机回退；
3. 如何用 `repeat_interleave(k, dim=1)` 正确实现 Video block 与 Action chunk 的 timestep coupling。

示例只操作 timestep ID，小张量、CPU 即可运行。

## 1. 构造与 DreamZero 训练相似的最小形状

假设每个样本包含 1 个条件 latent 帧和 4 个未来 latent 帧；未来帧每 2 帧组成一个 Video block。20 个 Action 对齐到 4 个未来 latent timestep，因此每个 latent timestep 对应 5 个 Action。

In [ ]:
import torch


torch.manual_seed(7)
torch.set_printoptions(linewidth=140)

BATCH_SIZE = 4
NUM_LATENT_FRAMES = 5          # 1 个条件帧 + 4 个未来帧
NUM_FUTURE_LATENT_FRAMES = NUM_LATENT_FRAMES - 1
NUM_FRAMES_PER_BLOCK = 2
ACTION_HORIZON = 20
ACTION_DIM = 7
NUM_TRAIN_TIMESTEPS = 1000

# 这里只需要形状；值本身不参与 timestep 构造。
actions = torch.zeros(BATCH_SIZE, ACTION_HORIZON, ACTION_DIM)
noise = torch.zeros(BATCH_SIZE, NUM_LATENT_FRAMES, 16, 2, 2) 
# video latent noise, (bs, 1 condition frame + 4 future frames, C, H, W)
noise_action = torch.randn_like(actions)

# 复现 Video timestep 采样与 block 对齐逻辑。
timestep_id = torch.randint(
    0,
    NUM_TRAIN_TIMESTEPS,
    (BATCH_SIZE, NUM_LATENT_FRAMES),
)
timestep_id_block = timestep_id[:, 1:].reshape(
    BATCH_SIZE,
    -1,
    NUM_FRAMES_PER_BLOCK,
)
timestep_id_block[:, :, 1:] = timestep_id_block[:, :, 0:1]
timestep_id_block = timestep_id_block.reshape(BATCH_SIZE, -1)

print("actions.shape             =", tuple(actions.shape))
print("noise.shape               =", tuple(noise.shape))
print("timestep_id_block.shape   =", tuple(timestep_id_block.shape))
print("timestep_id_block:\n", timestep_id_block)

assert timestep_id_block.shape == (
    BATCH_SIZE,
    NUM_FUTURE_LATENT_FRAMES,
)
assert torch.equal(timestep_id_block[:, 0], timestep_id_block[:, 1])
assert torch.equal(timestep_id_block[:, 2], timestep_id_block[:, 3])

actions.shape             = (4, 20, 7)
noise.shape               = (4, 5, 16, 2, 2)
timestep_id_block.shape   = (4, 4)
timestep_id_block:
 tensor([[794, 794,  10,  10],
        [459, 459, 903, 903],
        [372, 372, 828, 828],
        [818, 818, 880, 880]])


## 2. 解耦模式：Action 显式独立 `randint`

当 `decouple_video_action_noise=True` 时，Action timestep 与 Video 无关，直接为每个 Action 位置独立采样。

In [9]:
torch.manual_seed(11)
timestep_action_id_decoupled = torch.randint(
    0,
    NUM_TRAIN_TIMESTEPS,
    (actions.shape[0], actions.shape[1]),
)

print("decoupled action timestep shape =", tuple(timestep_action_id_decoupled.shape))
print("Video timesteps, sample 0       =", timestep_id_block[0].tolist())
print("Action timesteps, sample 0      =", timestep_action_id_decoupled[0].tolist())

assert timestep_action_id_decoupled.shape == (
    BATCH_SIZE,
    ACTION_HORIZON,
)

decoupled action timestep shape = (4, 20)
Video timesteps, sample 0       = [794, 794, 10, 10]
Action timesteps, sample 0      = [441, 679, 520, 27, 433, 247, 501, 20, 737, 847, 522, 760, 165, 204, 104, 885, 768, 605, 68, 581]


## 3. 当前 couple 分支：`.repeat(1, 1, k)` 触发随机回退

`timestep_id_block` 是二维 `[B, F]`，但传入三个 repeat 参数后，PyTorch 会在最前面补一个维度：

$$[B,F]\rightarrow[1,B,F]\rightarrow[1,B,Fk]$$

随后 reshape 仍无法恢复 batch 轴，形状检查失败，代码转而执行独立 `randint`。

In [ ]:
print(actions.shape, noise.shape) # torch.Size([4, 20, 7]) torch.Size([4, 5, 16, 2, 2])
actions_per_latent = (
    actions.shape[1] // (noise.shape[1] - 1)
    if (noise.shape[1] - 1) > 0
    else 1
)
print("actions_per_latent", actions_per_latent)

# 原代码的 couple 分支。
timestep_action_id_repeated = timestep_id_block.repeat(
    1,
    1,
    actions_per_latent,
) # (1, 1, 5)
timestep_action_id_reshaped = timestep_action_id_repeated.reshape(
    timestep_action_id_repeated.shape[0],
    -1,
) 
used_random_fallback = (
    timestep_action_id_reshaped.shape[1] != actions.shape[1]
)

if used_random_fallback:
    torch.manual_seed(12)
    timestep_action_id_current = torch.randint(
        0,
        NUM_TRAIN_TIMESTEPS,
        (actions.shape[0], actions.shape[1]),
    )
else:
    timestep_action_id_current = timestep_action_id_reshaped

print("actions_per_latent              =", actions_per_latent)
print("input timestep block shape      =", tuple(timestep_id_block.shape))
print("after repeat(1, 1, k) shape     =", tuple(timestep_action_id_repeated.shape))
print("after reshape shape             =", tuple(timestep_action_id_reshaped.shape))
print("expected action timestep shape  =", tuple(actions.shape[:2]))
print("used_random_fallback            =", used_random_fallback)
print("final current-code shape        =", tuple(timestep_action_id_current.shape))
print("final current-code sample 0     =", timestep_action_id_current[0].tolist())

assert timestep_action_id_repeated.shape == (
    1,
    BATCH_SIZE,
    ACTION_HORIZON,
)
assert timestep_action_id_reshaped.shape == (
    1,
    BATCH_SIZE * ACTION_HORIZON,
)
assert used_random_fallback is True
assert timestep_action_id_current.shape == (
    BATCH_SIZE,
    ACTION_HORIZON,
)

torch.Size([4, 20, 7]) torch.Size([4, 5, 16, 2, 2])
actions_per_latent 5
actions_per_latent              = 5
input timestep block shape      = (4, 4)
after repeat(1, 1, k) shape     = (1, 4, 20)
after reshape shape             = (1, 80)
expected action timestep shape  = (4, 20)
used_random_fallback            = True
final current-code shape        = (4, 20)
final current-code sample 0     = [363, 803, 222, 277, 393, 386, 219, 83, 988, 720, 374, 993, 852, 525, 949, 641, 82, 387, 558, 818]


## 4. 正确 couple：使用 `repeat_interleave()`

正确语义是把每个 Video latent timestep **连续重复**到对应 Action 位置，同时保持 batch 轴：

$$[B,F]\rightarrow[B,Fk]=[B,A]$$

In [21]:
if actions.shape[1] % timestep_id_block.shape[1] != 0:
    raise ValueError(
        "Action horizon 必须能被未来 latent timestep 数整除："
        f"{actions.shape[1]} % {timestep_id_block.shape[1]} != 0"
    )

repeats = actions.shape[1] // timestep_id_block.shape[1]
print(timestep_id_block)
timestep_action_id_coupled = timestep_id_block.repeat_interleave(
    repeats=repeats,
    dim=1,
)
print(timestep_action_id_coupled)

print("repeats                         =", repeats)
print("correct coupled shape           =", tuple(timestep_action_id_coupled.shape))
print("Video timesteps, sample 0       =", timestep_id_block[0].tolist())
print("Coupled Action IDs, sample 0    =", timestep_action_id_coupled[0].tolist())

assert timestep_action_id_coupled.shape == (
    BATCH_SIZE,
    ACTION_HORIZON,
)

# 每个 2-latent Video block 对齐 10 个 Action。
num_blocks = NUM_FUTURE_LATENT_FRAMES // NUM_FRAMES_PER_BLOCK
actions_per_block = ACTION_HORIZON // num_blocks
for batch_idx in range(BATCH_SIZE):
    for block_idx in range(num_blocks):
        expected_timestep = timestep_id_block[
            batch_idx,
            block_idx * NUM_FRAMES_PER_BLOCK,
        ]
        action_chunk = timestep_action_id_coupled[
            batch_idx,
            block_idx * actions_per_block:(block_idx + 1) * actions_per_block,
        ]
        assert torch.all(action_chunk == expected_timestep)

print("检查通过：每个 Action chunk 与对应 Video block 使用同一 timestep。")

tensor([[794, 794,  10,  10],
        [459, 459, 903, 903],
        [372, 372, 828, 828],
        [818, 818, 880, 880]])
tensor([[794, 794, 794, 794, 794, 794, 794, 794, 794, 794,  10,  10,  10,  10,  10,  10,  10,  10,  10,  10],
        [459, 459, 459, 459, 459, 459, 459, 459, 459, 459, 903, 903, 903, 903, 903, 903, 903, 903, 903, 903],
        [372, 372, 372, 372, 372, 372, 372, 372, 372, 372, 828, 828, 828, 828, 828, 828, 828, 828, 828, 828],
        [818, 818, 818, 818, 818, 818, 818, 818, 818, 818, 880, 880, 880, 880, 880, 880, 880, 880, 880, 880]])
repeats                         = 5
correct coupled shape           = (4, 20)
Video timesteps, sample 0       = [794, 794, 10, 10]
Coupled Action IDs, sample 0    = [794, 794, 794, 794, 794, 794, 794, 794, 794, 794, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
检查通过：每个 Action chunk 与对应 Video block 使用同一 timestep。


In [ ]:
a = torch.ones(4, 4)
col_n = a.new_full((4, 1), 2)
a = torch.cat([a, col_n], dim=1)
print("a after:", a)

a_r = a.repeat_interleave( # detect the same number, then multiply via timeline 
    repeats=4,
    dim=1
)
a_r

a after: tensor([[1., 1., 1., 1., 2.],
        [1., 1., 1., 1., 2.],
        [1., 1., 1., 1., 2.],
        [1., 1., 1., 1., 2.]])


tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 2., 2., 2., 2.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 2., 2., 2., 2.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 2., 2., 2., 2.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 2., 2., 2., 2.]])

## 5. 为什么 batch size = 1 也不能认为原实现正确

$B=1$ 时 shape 碰巧为 `[1, A]`，不会进入随机回退；但 `.repeat()` 会循环复制整段 timestep 序列，而不是连续复制每个 timestep，多 block 时数值顺序仍然错误。

In [5]:
single_sample_blocks = torch.tensor([[10, 10, 20, 20]])
current_batch1 = single_sample_blocks.repeat(1, 1, 5).reshape(1, -1)
correct_batch1 = single_sample_blocks.repeat_interleave(5, dim=1)

print("original block IDs =", single_sample_blocks.tolist())
print("current .repeat()  =", current_batch1.tolist())
print("correct result     =", correct_batch1.tolist())
print("same values/order? =", torch.equal(current_batch1, correct_batch1))

assert current_batch1.shape == correct_batch1.shape == (1, 20)
assert not torch.equal(current_batch1, correct_batch1)

original block IDs = [[10, 10, 20, 20]]
current .repeat()  = [[10, 10, 20, 20, 10, 10, 20, 20, 10, 10, 20, 20, 10, 10, 20, 20, 10, 10, 20, 20]]
correct result     = [[10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20]]
same values/order? = False


## 6. 可直接替换的安全实现

注意：couple 的是 **timestep / sigma**，不是高斯噪声。`noise_action = torch.randn_like(actions)` 在两种模式下都应独立生成。

In [6]:
def build_action_timestep_ids(
    *,
    actions: torch.Tensor,
    timestep_id_block: torch.Tensor,
    num_train_timesteps: int,
    decouple_video_action_noise: bool,
) -> torch.Tensor:
    if decouple_video_action_noise:
        return torch.randint(
            0,
            num_train_timesteps,
            (actions.shape[0], actions.shape[1]),
            device=timestep_id_block.device,
        )

    num_future_timesteps = timestep_id_block.shape[1]
    if actions.shape[1] % num_future_timesteps != 0:
        raise ValueError(
            "Action horizon 必须能被未来 latent timestep 数整除："
            f"{actions.shape[1]} % {num_future_timesteps} != 0"
        )

    return timestep_id_block.repeat_interleave(
        repeats=actions.shape[1] // num_future_timesteps,
        dim=1,
    )


safe_coupled_ids = build_action_timestep_ids(
    actions=actions,
    timestep_id_block=timestep_id_block,
    num_train_timesteps=NUM_TRAIN_TIMESTEPS,
    decouple_video_action_noise=False,
)
safe_decoupled_ids = build_action_timestep_ids(
    actions=actions,
    timestep_id_block=timestep_id_block,
    num_train_timesteps=NUM_TRAIN_TIMESTEPS,
    decouple_video_action_noise=True,
)

assert torch.equal(safe_coupled_ids, timestep_action_id_coupled)
assert safe_decoupled_ids.shape == safe_coupled_ids.shape == actions.shape[:2]

print("safe coupled shape   =", tuple(safe_coupled_ids.shape))
print("safe decoupled shape =", tuple(safe_decoupled_ids.shape))
print("全部断言通过。")

safe coupled shape   = (4, 20)
safe decoupled shape = (4, 20)
全部断言通过。


## 结论

| 模式 | Video timestep | Action timestep | 是否正确对齐 |
|---|---|---|---|
| 解耦模式 | 按 Video 策略采样 | 独立 `randint` | 不要求对齐，这是设计行为 |
| 当前 couple，$B>1$ | 按 Video 策略采样 | 形状错误后回退到独立 `randint` | 否，是 bug |
| 当前 couple，$B=1$ 且多 block | 按 Video 策略采样 | shape 正确但顺序循环重复 | 否，是 bug |
| 修正后的 couple | 按 Video 策略采样 | `repeat_interleave()` 继承对应 Video timestep | 是 |

因此，标准 couple 训练应保证同一个 Video block 和对应 Action chunk 使用相同的 timestep；Video 噪声与 Action 噪声本身仍是独立高斯噪声。